In [1]:
# Install dependencies (uncomment on a fresh environment / Colab):
# %pip install torch datasets numpy matplotlib sentencepiece
%matplotlib inline

English ↔ Uzbek Translation with a Seq2Seq Network and Attention
==================================================================

Adapted from the PyTorch *NLP From Scratch* seq2seq tutorial (Sean Robertson),
re-pointed from French→English onto **English↔Uzbek** using the
[Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100/viewer/en-uz)
`en-uz` parallel corpus.

**We train two separate models** — one per direction:

- `en-uz`: English → Uzbek
- `uz-en`: Uzbek → English

Each is its own encoder/decoder with its own source and target vocabularies, which
is the simplest and most reliable setup for this GRU + Bahdanau-attention
architecture. Both checkpoints are saved to `checkpoints/en2uz.pt` and
`checkpoints/uz2en.pt` for later use in the app.

The architecture matches the lecture (GRU encoder + GRU decoder + Bahdanau
attention). The data pipeline is upgraded for quality:

- **Subword tokenization with SentencePiece (BPE)** — the original word-level
  vocab was a poor fit for Uzbek, which is agglutinative (a single "word" can
  carry many suffixes). Subwords share roots across morphological variants and
  remove almost all `<unk>` tokens.
- The full OPUS-100 corpus (~260k pairs) is used by default instead of a 50k cap.
- `MAX_LENGTH` is raised to 50 subword tokens.
- Training uses gradient clipping and scheduled sampling (the decoder sometimes
  sees its own prediction instead of the gold token) — both address common
  RNN-MT instability and exposure bias.

In [2]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Loading data files
==================

We pull the English↔Uzbek parallel data straight from the Hugging Face
`datasets` hub — [Helsinki-NLP/opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100/viewer/en-uz),
config `en-uz`. The first run downloads and caches it locally; later runs are instant.

Each example looks like `{"translation": {"en": "...", "uz": "..."}}`. We read the
`train` split into a single list of `(english, uzbek)` pairs and reorder per
direction later.

Similar to the character encoding used in the character-level RNN
tutorials, we will be representing each word in a language as a one-hot
vector, or giant vector of zeros except for a single one (at the index
of the word). Compared to the dozens of characters that might exist in a
language, there are many many more words, so the encoding vector is much
larger. We will however cheat a bit and trim the data to only use a few
thousand words per language.

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/word-encoding.png)


We\'ll need a unique index per word to use as the inputs and targets of
the networks later. To keep track of all this we will use a helper class
called `Lang` which has word → index (`word2index`) and index → word
(`index2word`) dictionaries, as well as a count of each word
`word2count` which will be used to replace rare words later.


In [3]:
PAD_token = 0
SOS_token = 1
EOS_token = 2
UNK_token = 3

# Uzbek is agglutinative: a word like `kitobimdan` ("from my book") packs root
# + possessive + ablative into one token. Word-level vocab explodes on its long
# tail and most rare forms collapse to <unk>. We use SentencePiece (BPE) so the
# tokenizer learns shared subword pieces across morphological variants.

import io
import os
import tempfile
import sentencepiece as spm


class SPTokenizer:
    """Drop-in stand-in for the original Lang class, backed by SentencePiece."""

    def __init__(self, name, sp_model_bytes):
        self.name = name
        self.sp_model_bytes = sp_model_bytes
        self.sp = spm.SentencePieceProcessor()
        self.sp.LoadFromSerializedProto(sp_model_bytes)
        self.n_words = self.sp.GetPieceSize()
        assert self.sp.pad_id() == PAD_token
        assert self.sp.bos_id() == SOS_token
        assert self.sp.eos_id() == EOS_token
        assert self.sp.unk_id() == UNK_token

    def encode(self, text):
        return self.sp.EncodeAsIds(text)

    def decode(self, ids):
        clean = [int(i) for i in ids
                 if int(i) not in (PAD_token, SOS_token, EOS_token)]
        return self.sp.DecodeIds(clean)

    def id_to_piece(self, i):
        return self.sp.IdToPiece(int(i))


def train_sentencepiece(texts, vocab_size, model_type="bpe",
                        input_sentence_size=200_000):
    """Train a SentencePiece BPE model. Returns raw model bytes.

    We write `texts` to a temporary file and pass `input=<path>` rather than
    `sentence_iterator=`. The iterator path streams each sentence across the
    Python <-> C++ (SWIG) boundary one at a time and is dramatically slower
    for large corpora — on ~500k sentences it can take 30+ minutes vs ~2 min
    for the file path. We also cap with `input_sentence_size` so BPE doesn't
    have to look at every sentence; a random subsample gives an essentially
    identical vocabulary.
    """
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False,
                                     encoding="utf-8") as f:
        for t in texts:
            f.write(t.replace("\n", " "))
            f.write("\n")
        corpus_path = f.name

    try:
        buf = io.BytesIO()
        spm.SentencePieceTrainer.Train(
            input=corpus_path,
            model_writer=buf,
            vocab_size=vocab_size,
            model_type=model_type,
            character_coverage=1.0,
            pad_id=PAD_token, bos_id=SOS_token, eos_id=EOS_token, unk_id=UNK_token,
            pad_piece="<pad>", bos_piece="<s>", eos_piece="</s>", unk_piece="<unk>",
            num_threads=4,
            input_sentence_size=input_sentence_size,
            shuffle_input_sentence=True,
        )
        return buf.getvalue()
    finally:
        os.unlink(corpus_path)

Uzbek (Latin script) uses the modifier letters in `oʻ`/`gʻ` and an apostrophe,
so unlike the original tutorial we do **not** force the text down to ASCII (that
would destroy meaningful characters). Instead we lowercase, normalize the various
apostrophe glyphs to a plain `'`, pad punctuation so it tokenizes as its own token,
and drop everything that isn't a word character or basic punctuation.

In [ ]:
# Keep Unicode word characters + the Uzbek apostrophe; pad punctuation.
# Also transliterate Uzbek Cyrillic -> Latin, because OPUS-100 mixes both
# scripts and the tokenizer would otherwise learn two disjoint vocabularies
# for the same words.

# Modern Uzbek Cyrillic -> Latin char map. This is a simple char-by-char
# substitution (not perfect orthography — e.g. word-initial "е" arguably
# becomes "ye"), but it's deterministic, which is all the tokenizer needs.
_UZ_CYRL_MAP = {
    "а": "a",  "б": "b",  "в": "v",  "г": "g",  "д": "d",
    "е": "e",  "ё": "yo", "ж": "j",  "з": "z",  "и": "i",
    "й": "y",  "к": "k",  "л": "l",  "м": "m",  "н": "n",
    "о": "o",  "п": "p",  "р": "r",  "с": "s",  "т": "t",
    "у": "u",  "ф": "f",  "х": "x",  "ц": "s",  "ч": "ch",
    "ш": "sh", "ъ": "'",  "ы": "i",  "ь": "",
    "э": "e",  "ю": "yu", "я": "ya",
    "ў": "o'", "қ": "q",  "ғ": "g'", "ҳ": "h",
}

def _uz_cyrl_to_latin(s):
    return "".join(_UZ_CYRL_MAP.get(c, c) for c in s)


def normalizeString(s):
    s = unicodedata.normalize('NFC', s.lower().strip())
    # unify the apostrophe glyphs Uzbek uses (oʻ / o’ / o‘ -> o')
    for a in ('ʻ', 'ʼ', '‘', '’', '`'):
        s = s.replace(a, "'")
    s = _uz_cyrl_to_latin(s)                            # Cyrillic -> Latin
    s = re.sub(r"([.!?,;:])", r" \1 ", s)               # split off punctuation
    s = re.sub(r"[^\w'.!?,;:]+", " ", s, flags=re.UNICODE)  # drop other symbols
    s = re.sub(r"\s+", " ", s)
    return s.strip()

We read the corpus once into a list of `(en, uz)` pairs. `prepareData` then
orders each pair for the requested direction and builds the two `Lang`
vocabularies.

In [5]:
from datasets import load_dataset

def load_opus_pairs(split="train", max_pairs=None):
    """Return a list of normalized (english, uzbek) pairs from OPUS-100 en-uz."""
    print(f"Loading Helsinki-NLP/opus-100 en-uz [{split}] ...")
    ds = load_dataset("Helsinki-NLP/opus-100", "en-uz", split=split)
    pairs = []
    for ex in ds:
        t = ex["translation"]
        en = normalizeString(t["en"])
        uz = normalizeString(t["uz"])
        if en and uz:
            pairs.append((en, uz))
        if max_pairs and len(pairs) >= max_pairs:
            break
    print(f"Kept {len(pairs)} non-empty pairs")
    return pairs

/home/mardon/Documents/rnn-translator-app/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence length budget. `MAX_LENGTH` is the cap on **subword tokens** per
sentence (EOS fits within it). With BPE this is much more generous than 15
words — each word averages 1.5–2 subwords. `MAX_WORDS` is a cheap pre-filter we
apply before tokenizing, just to drop the obviously-too-long pairs.

In [6]:
MAX_LENGTH = 50   # max subword tokens per sentence (EOS fits inside this)
MAX_WORDS = 60    # rough word-count cap, applied before tokenization

def filterPair(p):
    return len(p[0].split(' ')) < MAX_WORDS and \
        len(p[1].split(' ')) < MAX_WORDS

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Full data preparation for one direction:

-   Order each `(en, uz)` pair for the chosen direction
-   Filter by length
-   Train a SentencePiece (BPE) tokenizer on each side of the corpus

In [7]:
def prepareData(direction, base_pairs, vocab_size=8000):
    """direction: 'en-uz' or 'uz-en'.  base_pairs: list of (en, uz)."""
    if direction == "en-uz":
        src_name, tgt_name = "en", "uz"
        pairs = [(en, uz) for (en, uz) in base_pairs]
    elif direction == "uz-en":
        src_name, tgt_name = "uz", "en"
        pairs = [(uz, en) for (en, uz) in base_pairs]
    else:
        raise ValueError("direction must be 'en-uz' or 'uz-en'")

    print(f"[{direction}] read {len(pairs)} pairs")
    pairs = filterPairs(pairs)
    print(f"[{direction}] {len(pairs)} pairs after word-count filter (MAX_WORDS={MAX_WORDS})")

    print(f"[{direction}] training SentencePiece (vocab={vocab_size}) on {src_name} ...")
    src_bytes = train_sentencepiece((p[0] for p in pairs), vocab_size=vocab_size)
    print(f"[{direction}] training SentencePiece (vocab={vocab_size}) on {tgt_name} ...")
    tgt_bytes = train_sentencepiece((p[1] for p in pairs), vocab_size=vocab_size)

    input_lang = SPTokenizer(src_name, src_bytes)
    output_lang = SPTokenizer(tgt_name, tgt_bytes)
    print(f"[{direction}] vocab: {input_lang.name}={input_lang.n_words} "
          f"{output_lang.name}={output_lang.n_words}")
    return input_lang, output_lang, pairs

The Seq2Seq Model
=================

A Recurrent Neural Network, or RNN, is a network that operates on a
sequence and uses its own output as input for subsequent steps.

A [Sequence to Sequence network](https://arxiv.org/abs/1409.3215), or
seq2seq network, or [Encoder Decoder
network](https://arxiv.org/pdf/1406.1078v3.pdf), is a model consisting
of two RNNs called the encoder and decoder. The encoder reads an input
sequence and outputs a single vector, and the decoder reads that vector
to produce an output sequence.

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/seq2seq.png)

Unlike sequence prediction with a single RNN, where every input
corresponds to an output, the seq2seq model frees us from sequence
length and order, which makes it ideal for translation between two
languages.

Consider the sentence `Je ne suis pas le chat noir` →
`I am not the black cat`. Most of the words in the input sentence have a
direct translation in the output sentence, but are in slightly different
orders, e.g. `chat noir` and `black cat`. Because of the `ne/pas`
construction there is also one more word in the input sentence. It would
be difficult to produce a correct translation directly from the sequence
of input words.

With a seq2seq model the encoder creates a single vector which, in the
ideal case, encodes the \"meaning\" of the input sequence into a single
vector --- a single point in some N dimensional space of sentences.


The Encoder
===========

The encoder of a seq2seq network is a RNN that outputs some value for
every word from the input sentence. For every input word the encoder
outputs a vector and a hidden state, and uses the hidden state for the
next input word.

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/encoder-network.png)


In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

The Decoder
===========

The decoder is another RNN that takes the encoder output vector(s) and
outputs a sequence of words to create the translation.


Simple Decoder
==============

In the simplest seq2seq decoder we use only last output of the encoder.
This last output is sometimes called the *context vector* as it encodes
context from the entire sequence. This context vector is used as the
initial hidden state of the decoder.

At every step of decoding, the decoder is given an input token and
hidden state. The initial input token is the start-of-string `<SOS>`
token, and the first hidden state is the context vector (the encoder\'s
last hidden state).

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/decoder-network.png)


In [9]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

I encourage you to train and observe the results of this model, but to
save space we\'ll be going straight for the gold and introducing the
Attention Mechanism.


Attention Decoder
=================

If only the context vector is passed between the encoder and decoder,
that single vector carries the burden of encoding the entire sentence.

Attention allows the decoder network to \"focus\" on a different part of
the encoder\'s outputs for every step of the decoder\'s own outputs.
First we calculate a set of *attention weights*. These will be
multiplied by the encoder output vectors to create a weighted
combination. The result (called `attn_applied` in the code) should
contain information about that specific part of the input sequence, and
thus help the decoder choose the right output words.

![](https://i.imgur.com/1152PYf.png)

Calculating the attention weights is done with another feed-forward
layer `attn`, using the decoder\'s input and hidden state as inputs.
Because there are sentences of all sizes in the training data, to
actually create and train this layer we have to choose a maximum
sentence length (input length, for encoder outputs) that it can apply
to. Sentences of the maximum length will use all the attention weights,
while shorter sentences will only use the first few.

![](https://pytorch.org/tutorials/_static/img/seq-seq-images/attention-decoder-network.png)

Bahdanau attention, also known as additive attention, is a commonly used
attention mechanism in sequence-to-sequence models, particularly in
neural machine translation tasks. It was introduced by Bahdanau et al.
in their paper titled [Neural Machine Translation by Jointly Learning to
Align and Translate](https://arxiv.org/pdf/1409.0473.pdf). This
attention mechanism employs a learned alignment model to compute
attention scores between the encoder and decoder hidden states. It
utilizes a feed-forward neural network to calculate alignment scores.

However, there are alternative attention mechanisms available, such as
Luong attention, which computes attention scores by taking the dot
product between the decoder hidden state and the encoder hidden states.
It does not involve the non-linear transformation used in Bahdanau
attention.

In this tutorial, we will be using Bahdanau attention. However, it would
be a valuable exercise to explore modifying the attention mechanism to
use Luong attention.


In [10]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None,
                teacher_forcing_ratio=0.5):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            use_teacher = (target_tensor is not None
                           and random.random() < teacher_forcing_ratio)
            if use_teacher:
                # Teacher forcing: feed the gold token as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                # Scheduled sampling (or free run at eval): use the model's own pred
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>There are other forms of attention that work around the lengthlimitation by using a relative position approach. Read about "localattention" in <a href="https://arxiv.org/abs/1508.04025">Effective Approaches to Attention-based Neural MachineTranslation</a>.</p>

</div>

Training
========

Preparing Training Data
-----------------------

To train, for each pair we will need an input tensor (indexes of the
words in the input sentence) and target tensor (indexes of the words in
the target sentence). While creating these vectors we will append the
EOS token to both sequences.


In [11]:
def indexesFromSentence(lang, sentence):
    return lang.encode(sentence)

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def get_dataloader(batch_size, input_lang, output_lang, pairs):
    # Encode pairs with SentencePiece; drop any that overflow MAX_LENGTH.
    kept_src, kept_tgt = [], []
    over = 0
    for src, tgt in pairs:
        si = indexesFromSentence(input_lang, src) + [EOS_token]
        ti = indexesFromSentence(output_lang, tgt) + [EOS_token]
        if len(si) > MAX_LENGTH or len(ti) > MAX_LENGTH:
            over += 1
            continue
        kept_src.append(si)
        kept_tgt.append(ti)
    print(f"  kept {len(kept_src)} pairs after subword length filter "
          f"(dropped {over} > MAX_LENGTH={MAX_LENGTH})")

    n = len(kept_src)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int64)   # 0 == PAD_token
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int64)
    for i, (si, ti) in enumerate(zip(kept_src, kept_tgt)):
        input_ids[i, :len(si)] = si
        target_ids[i, :len(ti)] = ti

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    return DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

Training the Model
==================

To train we run the input sentence through the encoder, and keep track
of every output and the latest hidden state. Then the decoder is given
the `<SOS>` token as its first input, and the last hidden state of the
encoder as its first hidden state.

\"Teacher forcing\" is the concept of using the real target outputs as
each next input, instead of using the decoder\'s guess as the next
input. Using teacher forcing causes it to converge faster but [when the
trained network is exploited, it may exhibit
instability](http://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.378.4095&rep=rep1&type=pdf).

You can observe outputs of teacher-forced networks that read with
coherent grammar but wander far from the correct translation
-intuitively it has learned to represent the output grammar and can
\"pick up\" the meaning once the teacher tells it the first few words,
but it has not properly learned how to create the sentence from the
translation in the first place.

Because of the freedom PyTorch\'s autograd gives us, we can randomly
choose to use teacher forcing or not with a simple if statement. Turn
`teacher_forcing_ratio` up to use more of it.


In [12]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion, clip=1.0):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        # Gradient clipping — without this, RNN training easily diverges.
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), clip)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), clip)

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

This is a helper function to print time elapsed and estimated time
remaining given the current time and progress %.


In [13]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

The whole training process looks like this:

-   Start a timer
-   Initialize optimizers and criterion
-   Create set of training pairs
-   Start empty losses array for plotting

Then we call `train` many times and occasionally print the progress (%
of examples, time so far, estimated time) and average loss.


In [14]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=PAD_token)  # don't train on padding

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

Plotting results
================

Plotting is done with matplotlib, using the array of loss values
`plot_losses` saved while training.


In [15]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

Evaluation
==========

Evaluation is mostly the same as training, but there are no targets so
we simply feed the decoder\'s predictions back to itself for each step.
Every time it predicts a word we add it to the output string, and if it
predicts the EOS token we stop there. We also store the decoder\'s
attention outputs for display later.


In [16]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        sentence = normalizeString(sentence)
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze().tolist()
        if isinstance(decoded_ids, int):
            decoded_ids = [decoded_ids]
        if EOS_token in decoded_ids:
            decoded_ids = decoded_ids[:decoded_ids.index(EOS_token)]
        text = output_lang.decode(decoded_ids)
        pieces = [output_lang.id_to_piece(i) for i in decoded_ids
                  if i not in (PAD_token, SOS_token)] + ['<EOS>']
    return pieces, text, decoder_attn

We can evaluate random sentences from the training set and print out the
input, target, and output to make some subjective quality judgements:


In [17]:
def evaluateRandomly(encoder, decoder, input_lang, output_lang, pairs, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        _, output_text, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        print('<', output_text)
        print('')

Training and Evaluating
=======================

We train **both directions** below. Each call to `build_and_train` prepares the
data for that direction, builds a fresh encoder/decoder, and trains it.

Knobs:

- `MAX_PAIRS` caps how much of the ~260k-pair corpus we use. `None` = use all.
  Start with a smaller cap (e.g. 30000) for a quick sanity check.
- `vocab_size` is the size of each SentencePiece vocabulary. 8000 is a good
  default; raise to 16000 if you have more data and compute.
- `hidden_size`, `batch_size`, `n_epochs` are the usual capacity/throughput dials.

In [18]:
hidden_size = 256
batch_size = 128
n_epochs = 15
vocab_size = 8000
MAX_PAIRS = None    # use the whole OPUS-100 en-uz corpus (~260k pairs)

# Load the parallel corpus once (cached by the `datasets` library after first run).
base_pairs = load_opus_pairs(split="train", max_pairs=MAX_PAIRS)

# direction -> (encoder, decoder, input_lang, output_lang, pairs)
models = {}

def build_and_train(direction):
    input_lang, output_lang, pairs = prepareData(direction, base_pairs, vocab_size=vocab_size)
    dataloader = get_dataloader(batch_size, input_lang, output_lang, pairs)

    encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
    decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

    print(f"\n[{direction}] training for {n_epochs} epochs on {device} ...")
    train(dataloader, encoder, decoder, n_epochs, print_every=1, plot_every=1)

    models[direction] = (encoder, decoder, input_lang, output_lang, pairs)
    return models[direction]

# Two separate models, one per direction.
build_and_train("en-uz")
build_and_train("uz-en")

Loading Helsinki-NLP/opus-100 en-uz [train] ...
Kept 173110 non-empty pairs
[en-uz] read 173110 pairs
[en-uz] 171175 pairs after word-count filter (MAX_WORDS=60)
[en-uz] training SentencePiece (vocab=8000) on en ...


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /tmp/tmp3_k4_byf.txt
  input_format: 
  model_prefix: 
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 200000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 4
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differentia

[en-uz] training SentencePiece (vocab=8000) on uz ...


ize=3520 all=46829 active=2423 piece=▁қўйса
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=210 size=3540 all=46904 active=2498 piece=▁қанчалик
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=207 size=3560 all=46989 active=2583 piece=▁ўлиб
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=206 size=3580 all=47016 active=2610 piece=▁қилинмас
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=205 size=3600 all=47093 active=2687 piece=▁қалбларига
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=205 min_freq=98
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=203 size=3620 all=47239 active=2501 piece=▁келмаган
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=201 size=3640 all=47297 active=2559 piece=▁дарҳол
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=199 size=3660 all=47389 active=2651 piece=▁жонни
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=197 size=3680 all=47445 active=2707 piece=▁ойни
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=195 size=3700 all=47529 active=27

[en-uz] vocab: en=8000 uz=8000
  kept 164078 pairs after subword length filter (dropped 7097 > MAX_LENGTH=50)

[en-uz] training for 15 epochs on cuda ...
2m 19s (- 32m 29s) (1 6%) 5.9360
4m 38s (- 30m 10s) (2 13%) 4.7843
6m 55s (- 27m 40s) (3 20%) 4.1835


KeyboardInterrupt: 

Saving the models
=================

Each direction is saved as a self-contained checkpoint (weights + both
vocabularies + config) so the app can load it without re-reading the corpus.

In [ ]:
from pathlib import Path

def save_model(direction, path=None):
    encoder, decoder, input_lang, output_lang, _ = models[direction]
    path = Path(path or f"checkpoints/{direction.replace('-', '2')}.pt")
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "direction": direction,
        "hidden_size": hidden_size,
        "max_length": MAX_LENGTH,
        "encoder_state": encoder.state_dict(),
        "decoder_state": decoder.state_dict(),
        # SentencePiece model bytes — checkpoint is self-contained.
        "input_sp_model": input_lang.sp_model_bytes,
        "output_sp_model": output_lang.sp_model_bytes,
        "input_lang_name": input_lang.name,
        "output_lang_name": output_lang.name,
    }, path)
    print("saved", path)

save_model("en-uz")
save_model("uz-en")

Set dropout layers to `eval` mode


In [ ]:
for direction in ("en-uz", "uz-en"):
    encoder, decoder, input_lang, output_lang, pairs = models[direction]
    encoder.eval()
    decoder.eval()
    print(f"\n===== {direction} =====")
    evaluateRandomly(encoder, decoder, input_lang, output_lang, pairs, n=5)

Visualizing Attention
=====================

A useful property of the attention mechanism is its highly interpretable
outputs. Because it is used to weight specific encoder outputs of the
input sequence, we can imagine looking where the network is focused most
at each time step.

You could simply run `plt.matshow(attentions)` to see attention output
displayed as a matrix. For a better viewing experience we will do the
extra work of adding axes and labels:


In [ ]:
def showAttention(input_pieces, output_pieces, attentions):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap='bone')
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticklabels([''] + input_pieces + ['<EOS>'], rotation=90)
    ax.set_yticklabels([''] + output_pieces)

    # Show label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()


# Visualize attention for the English -> Uzbek model.
enc_viz, dec_viz, in_viz, out_viz, _ = models["en-uz"]

def evaluateAndShowAttention(input_sentence):
    output_pieces, output_text, attentions = evaluate(
        enc_viz, dec_viz, input_sentence, in_viz, out_viz)
    print('input  =', input_sentence)
    print('output =', output_text)
    input_pieces = [in_viz.id_to_piece(i)
                    for i in in_viz.encode(normalizeString(input_sentence))]
    n_in = len(input_pieces) + 1  # +1 for the implicit <EOS> column
    n_out = len(output_pieces)
    showAttention(input_pieces, output_pieces, attentions[0, :n_out, :n_in])


evaluateAndShowAttention('i am very happy today')
evaluateAndShowAttention('where is the train station ?')
evaluateAndShowAttention('thank you very much')
evaluateAndShowAttention('she is reading a book')

Exercises
=========

-   Try with a different dataset
    -   Another language pair
    -   Human → Machine (e.g. IOT commands)
    -   Chat → Response
    -   Question → Answer
-   Replace the embeddings with pretrained word embeddings such as
    `word2vec` or `GloVe`
-   Try with more layers, more hidden units, and more sentences. Compare
    the training time and results.
-   If you use a translation file where pairs have two of the same
    phrase (`I am test \t I am test`), you can use this as an
    autoencoder. Try this:
    -   Train as an autoencoder
    -   Save only the Encoder network
    -   Train a new Decoder for translation from there
